# Lab 4 - Deep Learning for Computer Vision

Today we are going to use the TensorFlow/Keras library to build out own Computer Vision model, as well as fine-tune a pre-trained model.

In this lab, we will also learn how to save trained models and then load them for later use. **Saving and loading models will be important for your coursework!**

First, we will install the required libraries.

In [ ]:
!pip install -q tensorflow matplotlib numpy

After the correct libraries are installed, we will then import the parts that we are going to use today:

In [ ]:
print("Importing libraries...")
import os, time, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow version:", tf.__version__)
np.random.seed(123)
tf.random.set_seed(123)
random.seed(123)
print("Libraries imported!")

# Step 1 - Loading the Data

We are going to use the Cats vs Dogs dataset, available from TensorFlow datasets. Since all the images are different sizes, we will also resize them to 224px RGB images (producing a data size of 224 x 224 x 3).

While the dataset is downloading and preprocessing, read about the dataset here - [https://www.tensorflow.org/datasets/catalog/cats_vs_dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs)

**See if you can find out how many images are in the dataset** - Since the dataset is quite large, we are going to use a subset of 2,500 images. 2,000 for training and 500 for testing - you can increase these numbers during your own experimentation for a larger dataset.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64

print("Donwloading the full dataset...")
# Load the full dataset
(raw_train, raw_test), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    with_info=True,
    as_supervised=True,
)
print("Data downloaded.")

# Sometimes larger images cause Python to crash with TensorFlow Datasets if you have a limited amount of RAM
# To overcome this, we will store our images as standard arrays, instead.
def preprocess_to_numpy(dataset, n_samples):
    dataset = dataset.shuffle(10000, seed=42).take(n_samples)
    images, labels = [], []
    for image, label in tfds.as_numpy(dataset):
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE)).numpy()
        image = image.astype("float32") / 255.0
        images.append(image)
        labels.append(label)
    return np.array(images), np.array(labels)

print("Preprocessing training data...")
x_train, y_train = preprocess_to_numpy(raw_train, 2000)
print("Training data preprocessed.")
print("Preprocessing validation data...")
x_test, y_test   = preprocess_to_numpy(raw_test, 500)
print("Validation data preprocessed.")

print("Train:", x_train.shape, y_train.shape)
print("Validation :", x_test.shape, y_test.shape)

# Step 1.1 - Visualising the Images

Let's take a look at 12 random images from the dataset. Run this code multiple times to see various images from the dataset.

**Point of reflection** - do some of the cats and dogs look similar? Rather than considering the whole animal, think about features such as lines, textures, and shapes. For example, do any of the animals have similar fur? Which do you think will be the hardest for the CNN to recognise?

In [ ]:
plt.figure(figsize=(12, 6))

# pick 12 random images
idxs = np.random.choice(len(x_train), size=12, replace=False)

for i, idx in enumerate(idxs):
    plt.subplot(3, 4, i+1)
    plt.imshow(x_train[idx])  # already scaled to 0-1 floats
    plt.title("Dog" if y_train[idx] == 1 else "Cat")
    plt.axis("off")

plt.suptitle("Random Examples from the Dataset")
plt.tight_layout()
plt.show()

# Data Augmentation

Before we start, we will introduce some data augmentation. Below, I have included a random flip, rotation, and zoom.

Try adding your own layers to this data augmentation stack. A full list of the available layers can be found here - [https://keras.io/api/layers/preprocessing_layers/image_augmentation/](https://keras.io/api/layers/preprocessing_layers/image_augmentation/)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Step 2 - Creating a Simple CNN

In the below code, we will create a CNN that has 32 convolutions and 64, then the output.

Following the lecture material, this is a binary classification problem. Therefore, we will use the **Sigmoid** activation function for the output.

Run the code to build the simple CNN. It is then ready to train!

In [ ]:
def build_simple_cnn(input_shape=(224,224,3)):
    model = keras.Sequential([
        data_augmentation,
        layers.Conv2D(32, 3, activation="relu", padding="same", input_shape=input_shape),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model

simple_cnn = build_simple_cnn()
simple_cnn.compile(optimizer="adam",
                   loss="binary_crossentropy",
                   metrics=["accuracy"])
simple_cnn.summary()

# Step 2.2 - Training and Saving our CNN

Now that we have built our CNN, let's train it! We will also save the weights. This ensures that we can load our model for use later on.

For now, we will train it for 3 epochs with a batch size of 64.

*Epochs* means how many times the model will process each training image. Once all of the images have been passed through the model, the epoch has ended. 3 epochs therefore means that our model will process each training image a total of 3 times.

*Batch Size* means how many images are passed through the model before the weights are updated. 64 means that, at each step, 64 images are passed through the model before the weights are updated. Updating the weights means attempting to learn from the data.





In [ ]:
history_simple = simple_cnn.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs = 3,
    batch_size = 64
)

simple_cnn.save("simple_cnn.keras")

# Step 2.3 - Loading our CNN and Evaluating

In [ ]:
print("Loading saved CNN...")
loaded_simple = keras.models.load_model("simple_cnn.keras")
print("Saved CNN has been loaded!\n")

print("Evaluating the loaded CNN...")
simple_test_loss, simple_test_acc = loaded_simple.evaluate(x_test, y_test, verbose=0)
simple_test_acc = format(simple_test_acc*100, '.2f') # times by 100 and round to 2 decimal places
print("Simple CNN Test Accuracy: " + simple_test_acc + "%")

# Step 2.4 - Build your own CNN!

This next step is a copy and paste of the above code, except the layers have been removed (see the "add your layers here!" line of code).

Experiment by creating your own sequence of layers to create a new CNN. It will be saved as "my_cnn_cifar10". Ensure you do this as we will be comparing all of the models at the end!


In [ ]:
def build_my_cnn(input_shape=(32,32,3)):
    model = keras.Sequential([
        data_augmentation,
        layers.Conv2D(32, 3, activation="relu", padding="same", input_shape=input_shape), # you can change the number of filters and window size by changing 32 and 3, respectively
        layers.MaxPooling2D(),
        # add your layers here!
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model

my_cnn = build_my_cnn()
my_cnn.compile(optimizer="adam",
                   loss="binary_crossentropy",
                   metrics=["accuracy"])
my_cnn.summary()

ckpt_path_simple_dense = "my_cnn_cifar10.keras"

history_simple = my_cnn.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),  # use CIFAR-10’s official test set
    epochs=3,
    batch_size=64,
)

my_cnn.save("my_cnn.keras")

print("Loading your own CNN...")
loaded_my_cnn = keras.models.load_model("my_cnn.keras")
print("Your own CNN has been loaded!\n")

print("Evaluating the loaded CNN...")
my_cnn_test_loss, my_cnn_test_acc = loaded_my_cnn.evaluate(x_test, y_test, verbose=0)
my_cnn_test_acc = format(my_cnn_test_acc*100, '.2f') # times by 100 and round to 2 decimal places
print("Your CNN Test Accuracy: " + my_cnn_test_acc + "%")

# Step 3 - Transfer Learning with VGG16

We will now fine-tune the VGG16 model. As you will see, the model is quite large. It will take longer to train than our smaller models.

While you wait, read about VGG16 here:
*   [https://en.wikipedia.org/wiki/VGGNet](https://en.wikipedia.org/wiki/VGGNet)
*   [https://keras.io/api/applications/vgg/](https://keras.io/api/applications/vgg/)


In [ ]:
from tensorflow.keras.applications import VGG16

# Here we will download and load the VGG16 CNN trained on ImageNet
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))


# Since the model is large, so we won't train it all.
# Instead, we will freeze some of the weights and use those as a Feature Extractor.
# Below, we unfreeze the last 4 layers (except for the batch normalisation layers)
# Therefore, we are using both types of Transfer Learning (Feature Extraction early on, then Fine-tuning later in the model)
for layer in base_model.layers[:-4]:
    layer.trainable = False
for layer in base_model.layers[-4:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True


# We now need to attach a new output layer to the model. In our case, we have 10 classes
VGG16_model = keras.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation="sigmoid")
])


# Now we will compile and train the model!
VGG16_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
VGG16_model.summary()

ckpt_path_efficient_net = "efficient_net_cifar10.keras"

history_efficientnet = VGG16_model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=3,
    batch_size=64
)

VGG16_model.save("VGG16_model.keras")


print("Loading saved VGG16...")
loaded_vgg16 = keras.models.load_model("VGG16_model.keras")
print("VGG16 has been loaded!\n")

print("Evaluating VGG16...")
vgg16_test_loss, vgg16_test_acc = loaded_vgg16.evaluate(x_test, y_test, verbose=0)
vgg16_test_acc = format(vgg16_test_acc * 100, '.2f')
print("VGG16 Test Accuracy: " + vgg16_test_acc + "%")


# Step 4 - Comparing the Models

Now let's load all three models, run the evaluation again, and print out the results.

**Point of Reflection** - which model scored the highest? Why do you think that is?

In [ ]:
print("Running evaluations...\n")

# The tutorial CNN
loaded_simple = keras.models.load_model("simple_cnn.keras")
simple_test_loss, simple_test_acc = loaded_simple.evaluate(x_test, y_test, verbose=0)
simple_test_acc = format(simple_test_acc*100, '.2f') # times by 100 and round to 2 decimal places

# Your own CNN
loaded_my_cnn = keras.models.load_model("my_cnn.keras")
my_cnn_test_loss, my_cnn_test_acc = loaded_my_cnn.evaluate(x_test, y_test, verbose=0)
my_cnn_test_acc = format(my_cnn_test_acc*100, '.2f') # times by 100 and round to 2 decimal places

# VGG16
loaded_vgg16 = keras.models.load_model("VGG16_model.keras")
vgg16_test_loss, vgg16_test_acc = loaded_vgg16.evaluate(x_test, y_test, verbose=0)
vgg16_test_acc = format(vgg16_test_acc * 100, '.2f')  # times by 100 and round to 2 decimal places


print("Simple CNN Test Accuracy: " + simple_test_acc + "%")
print("Your own CNN Test Accuracy: " + my_cnn_test_acc + "%")
print("VGG16 Test Accuracy: " + vgg16_test_acc + "%")

# Step 5 - Making Predictions on Unseen Images

We will now use our three models to make some predictions on unseen photographs. I have provided four images in this week's NOW Learning room - a cat, a dog, a lion, and a wolf.

Run the below code, and each of the four images will have their class predicted by the model.

For cat.jpg and dog.jpg, the models should make sensible predictions (though not always correct). For the lion and the wolf, the model will **force** a decision from one of the two classes we trained it with.

This highlights a key limitation - deep learning models will **always** try to classify images into one of the known categories, even when the input does not belong to them!

**Point of reflection** - this has real-world consequences. In safety-critical domains such as medical imaging and autonomous vehicles, misclassifications outside of the training domain can be **dangerous**. Would you trust a model's output when classifying something it has never seen before? Think about how you might detect or handle unknown classes in practice.

# Try it yourself
Collect some images from the internet and save them in the same location as this notebook. Load them by modifying the code below and make predictions using the three models you have trained today.

**Interpreting the Sigmoid values** - following the prediction, we print out the sigmoid values to see how confident the model is. For this dataset, a cat is 0 and a dog is 1. A value closer to 0 means that the model is more confident that it is a cat, likewise for values closer to 1 and dogs. A value around 0.5 means that the model has low confidence for either class.

In [ ]:
def evaluate_image(img_path):
    print(f"\nEvaluating: {img_path}")

    # Load and preprocess image
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)   # add the batch dimension (1 in this case)
    img_array = img_array / 255.0                   # normalise all pixels to 0-1

    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path)
    plt.show()

    # Run predictions on all three models
    pred_simple = loaded_simple.predict(img_array, verbose=0)[0][0]
    pred_my_cnn = loaded_my_cnn.predict(img_array, verbose=0)[0][0]
    pred_vgg16 = loaded_vgg16.predict(img_array, verbose=0)[0][0]

    # Convert probabilities to class labels
    def to_label(pred):
        return "Dog" if pred > 0.5 else "Cat"

    print(f"Simple CNN Prediction: {to_label(pred_simple)} ({pred_simple:.2f})")
    print(f"My CNN Prediction: {to_label(pred_my_cnn)} ({pred_my_cnn:.2f})")
    print(f"VGG16 Prediction: {to_label(pred_vgg16)} ({pred_vgg16:.2f})")

# Example usage with your four test images
evaluate_image("cat.jpg")
evaluate_image("dog.jpg")
evaluate_image("lion.jpg")
evaluate_image("wolf.jpg")

# **Lab Challenge**

Now, over to you!

So far, have implemented three CNNs. A tutorial one that I created for you, your own CNN, and VGG16. Your task now is to implement another transfer learning model and compare it to the three that we have trained together. Ensure that you use a batch size of 64 and train it for 3 epochs so the results are directly comparable to our other models.

**You may wish to look back at the VGG16 code for guidance.**

**Easier Task** - Implement VGG19 ([https://keras.io/api/applications/vgg/](https://keras.io/api/applications/vgg/))

**More Challenging Task** - Implement another type of pre-trained CNN from the Keras applications - [https://keras.io/api/applications/](https://keras.io/api/applications/)

In [ ]:
# write your code here!